# OpenTelemetry (OTel) Manual Instrumentation Playground
This notebook demonstrates low-level manual instrumentation using OpenTelemetry Python API:
1. Configuring Tracer Providers and Span processors
2. Injecting custom attributes and event tags into span contexts
3. Recording exceptions and error statuses in spans.

In [1]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
from opentelemetry import trace
from opentelemetry.trace import StatusCode
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from ragforge.config import PHOENIX_COLLECTOR_URL

provider = TracerProvider()
processor = SimpleSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_COLLECTOR_URL))
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("notebook-otel-explorer")
print("Tracer provider successfully configured!")

Tracer provider successfully configured!


## 1. Instrument Operations & Log Custom Span Events

In [3]:
with tracer.start_as_current_span("QueryExecution") as parent_span:
    parent_span.set_attribute("db.system", "qdrant")
    parent_span.set_attribute("db.query", "search_documents")
    
    # Span Event annotation
    parent_span.add_event("Connection established")
    
    with tracer.start_as_current_span("VectorDistanceComputation") as child_span:
        child_span.set_attribute("vector.dimensions", 768)
        child_span.add_event("Distance calculation complete")
        
    parent_span.add_event("Results fetched successfully")

print("Manual tracing spans logged successfully!")

Manual tracing spans logged successfully!


## 2. Record Failures & Exceptions inside Spans

In [4]:
with tracer.start_as_current_span("NetworkRequestTool") as span:
    try:
        # Simulate a crash
        print("Simulating API request to server...")
        raise ConnectionError("Connection to server at localhost:8080 timed out.")
    except Exception as e:
        # Log exception in span details
        span.record_exception(e)
        span.set_status(StatusCode.ERROR, str(e))
        print("Exception details bound to OpenTelemetry span record.")

Simulating API request to server...
Exception details bound to OpenTelemetry span record.
